# Controllers on M2 with the reference known in advance

Compares, on `make_calibrated_sim(control_hz=100)` (0.5 s dead time, 0.6 s lag):

| Controller | What it is |
|---|---|
| Delay-aware SMC (M2, known ref) | `softarm.controllers.DelayAwareNDOSMC`, λ = 8, μ = 1 mm, no observer; reads r(t + T) directly |
| Delay-aware NDOSMC (M2, known ref) | same with the disturbance observer, Nikoletta's gains (λ = 4, l₁ = 0.1, Φ = 1) |
| Delay-aware SMC (M2, known ref), λ = 4 | the SMC at the NDOSMC's gain, no observer: compared with the NDOSMC it isolates the observer |
| MPC (M2), horizon H steps | `softarm.controllers.DelayMPC`: delay-compensated, unconstrained MPC on M2; H in {25, 100, 300} steps |
| MPC (F), horizon 300 steps = 3.00 s | `softarm.controllers.KalmanDelayMPC` on the frequency-identified model F (per axis, channels ν_x, ν_y), its own tuned weight ρ = 1e-5 m²/psi², given the true future reference |

The professor confirmed the final tests use random, irregular trajectories that are known in
advance, so every controller reads the future reference.

The controllers, tests and metrics are in the `softarm` package; the experiment (R_u tuning, runs,
figures) is `softarm.experiments.m2_comparison` (run as `scripts/compare_m2_controllers.py`). This notebook calls it and shows the results.
Runs offline in the project environment (`uv run jupyter lab`); about 30 s with 10 worker processes.

In [ ]:
import time
import numpy as np
from IPython.display import Markdown, display
from softarm import plotting as P
from softarm.constants import DT, FIG
from softarm.evaluation import TESTS
from softarm.experiments import m2_comparison as C

%matplotlib inline
P.apply()

## 1 — The MPC on M2

M2: state $X = (y, p)$, tip (mm) and measured S2–S4 pressure (psi), both relative to the settled pose;
input $u$ = command − 4.5 psi.

$$\tau \dot p = -p + G\,u(t - T), \qquad \dot y = -\Lambda y + \Lambda H p, \qquad T = 0.5\ \text{s}\ (N = 50\ \text{steps}),\ \tau \approx 0.6\ \text{s}$$

Every 10 ms step $k$:

1. Discretise M2 with zero-order hold: $X_{k+1} = A X_k + B u_{k-N}$.
2. Predict across the dead time with the commands already sent: $X_{k+N} = A^N X_k + \sum_{i=0}^{N-1} A^{N-1-i} B\, u_{k-N+i}$.
3. Model-error offset $w$: measured tip now minus the tip predicted for now $N$ steps ago, low-pass (0.5 s).
4. Minimise
$$J = \sum_{i=0}^{H-1} \lVert C X_{k+N+1+i} + w - r_{k+N+i} \rVert^2 + R_u \sum_{i=0}^{H-1} \lVert u_{k+i} - u_{k+i-1} \rVert^2$$
   in closed form (no constraints), apply the first move, clip to 0–9 psi.

**Horizon** $H$ counts 10 ms steps *after* the dead time: $H$ = 25 steps = 0.25 s looks 0.5–0.75 s ahead,
100 steps = 1 s looks 0.5–1.5 s ahead, 300 steps = 3 s looks 0.5–3.5 s ahead.

$R_u$ (mm²/psi²) is chosen per horizon on a separate tuning circle (5 mm, 0.07 Hz, seed 5), never on a test.

In [ ]:
t0 = time.perf_counter()
r_u, tuned = C.tune()

rows = ["| Horizon | " + " | ".join(f"R_u = {r:g}" for r in C.RU_GRID) + " | chosen |", "|---" * (len(C.RU_GRID) + 2) + "|"]
for h in C.HORIZONS:
    e = {r: v for hh, r, v in tuned if hh == h}
    rows.append(f"| {h} steps = {h * DT:.2f} s | " + " | ".join(f"{e[r]:.3f}" for r in C.RU_GRID) + f" | **{r_u[h]:g}** |")
display(Markdown("**R_u tuning** (tuning circle; RMSE after the first lap, mm)\n\n" + "\n".join(rows)))

## 2 — Tests

| Test | Reference | Sim seed | "After settling" starts |
|---|---|---|---|
| held-out circle | 10 mm, 0.15 Hz | 5 | after the first lap (6.7 s) |
| scored circle | 7 mm, 0.1 Hz (package `CircularTrackingTask`) | 0 | after the first lap (10 s) |
| squiggle | per axis 6 random sines, 0.03–0.3 Hz, ±6 mm, path seed 101 | 0 | 5 s |
| random waypoints | straight lines between random points, ±6 mm, 1–2.5 s each, path seed 101 | 0 | 5 s |
| triangle | vertices on a 7 mm circle, first at +x, 10 s per lap (as `motion_tests.py`) | 0 | after the first lap (10 s) |
| square | axis-aligned, vertices on a 7 mm circle (side 9.9 mm), 10 s per lap | 0 | after the first lap (10 s) |

All 30 s, after the harness's settle. Metrics, planar x–y, mm:

- **RMSE whole run**: what the package harness reports; includes the start-up jump.
- **RMSE after settling**: the same error once the start-up transient is over.
- **Shape error**: RMS distance from the tip to the *nearest point* of the reference path after settling;
  ignores timing, so it measures how closely the traced curve conforms to the shape.
- **Max error** after settling, **radius range** of the traced circle, **path size** of the triangle and square
  (RMS distance from the centre, traced / reference; 100 % = right size, > 100 % = overshoot), **saturation** (% of commands at 0 or 9 psi),
  **lag** of y behind the reference.

In [ ]:
runs = C.run_all(r_u)
print(f"{len(runs)} runs in {time.perf_counter() - t0:.0f} s")
R = {(r["test"], r["controller"]): r["metrics"] for r in runs}
MAIN = (["Delay-aware SMC (M2, known ref)", C.SMC_L4, "Delay-aware NDOSMC (M2, known ref)"] + [C.mpc_label(h) for h in C.HORIZONS]
        + [C.MPC_F])


def table(test: str, names=MAIN) -> None:
    kind = TESTS[test][3]
    after = "after 5 s" if kind == "open" else "after first lap"
    head = ["Controller", "RMSE whole run", f"RMSE {after}", "Shape error", f"Max {after}", "Saturation %", "Lag (s)"]
    head += {"circle": ["Radius range"], "polygon": ["Path size %"], "open": []}[kind]
    out = ["| " + " | ".join(head) + " |", "|---" * len(head) + "|"]
    for n in names:
        m = R[(test, n)]
        cells = [n, f"{m['rmse_whole_mm']:.2f}", f"{m['rmse_settled_mm']:.3f}", f"{m['shape_mm']:.3f}",
                 f"{m['max_settled_mm']:.2f}", f"{m['saturated_pct']:.1f}", f"{m['lag_s']:+.2f}"]
        if kind == "polygon":
            cells.append(f"{m['size_pct']:.0f}")
        if kind == "circle":
            cells.append(f"{m['radius_min_mm']:.1f}–{m['radius_max_mm']:.1f} (ref {m['radius_ref_mm']:.1f})")
        out.append("| " + " | ".join(cells) + " |")
    display(Markdown(f"**{test}** (mm)\n\n" + "\n".join(out)))

## 3 — Results per test

Each figure: left, the tip path in x–y against the reference (last lap for circles, after 5 s otherwise);
right, the planar error over time. Top row: SMC, NDOSMC, the tuned MPC on M2 and the MPC on F, all with the
known reference. Bottom row: the three M2 MPC horizons at R_u = 10, where the horizon matters (section 4).

### Held-out circle

In [ ]:
table("held-out circle")
C.figure(runs, "held-out circle");

### Scored circle

In [ ]:
table("scored circle")
C.figure(runs, "scored circle");

### Triangle

In [ ]:
table("triangle")
C.figure(runs, "triangle");

### Square

In [ ]:
table("square")
C.figure(runs, "square");

### Squiggle (smooth random path)

In [ ]:
table("squiggle")
C.figure(runs, "squiggle");

### Random waypoints (straight lines, sharp corners)

In [ ]:
table("random waypoints")
C.figure(runs, "random waypoints");

## 4 — Horizon sensitivity

At the tuned (very small) R_u the MPC is close to a model inverse and only the first steps of the
horizon affect the command, so all horizons give the same result. Penalising command changes makes the
horizon matter. RMSE after settling, mm:

In [ ]:
head = ["R_u", "Horizon"] + list(TESTS)
out = ["| " + " | ".join(head) + " |", "|---" * len(head) + "|"]
for ru in (None,) + C.RU_FIXED:
    for h in C.HORIZONS:
        n = C.mpc_label(h, ru)
        out.append(f"| {r_u[h] if ru is None else ru:g}{' (tuned)' if ru is None else ''} | {h} steps = {h * DT:.2f} s | "
                   + " | ".join(f"{R[(t, n)]['rmse_settled_mm']:.3f}" for t in TESTS) + " |")
display(Markdown("\n".join(out)))

## 5 — Save (optional)

Writes `docs/figures/m2_controllers.json` (every run: controller, gains, test, metrics, tip and reference
traces every 50 ms) and `docs/figures/m2ctrl_<test>.png`, the same files as
`uv run python scripts/compare_m2_controllers.py`.

In [ ]:
SAVE = False
if SAVE:
    C.save(runs, r_u, tuned, time.perf_counter() - t0)
    for t in TESTS:
        C.figure(runs, t, FIG / f"m2ctrl_{t.replace(' ', '_')}.png")